# 04 — Breath Image Dataset

Generates a standardised image dataset from raw breath waveforms for image classification (FL vs NFL).

## Standardisation approach
- **Amplitude**: normalised by PIF (peak inspiratory flow) so all waveforms sit on the same scale
- **Time axis**: each breath resampled to exactly `N_SAMPLES` points (linear interpolation), so every image represents the same fractional progress through the breath
- **Y-axis**: fixed across all images (computed from global percentiles of the training set) so visual area encodes the same flow range in every image
- **Canvas**: `IMG_SIZE × IMG_SIZE` pixels, white background, no axes/ticks/labels/borders — just the waveform line

## Output
```
breath_images/
  FL/   {participant}_breath_{N}.png
  NFL/  {participant}_breath_{N}.png
```

## 1. Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — faster for bulk saving
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH   = Path('./data')
OUT_ROOT    = Path('./breath_images')
IMG_SIZE    = 224          # pixels (square) — standard for CNN / transfer learning
N_SAMPLES   = 300          # resample every breath to this many points
LINE_WIDTH  = 2.5          # waveform line width in points
LINE_COLOR  = 'black'
LABELS_USE  = {'FL', 'NFL'}
MIN_SAMPLES = 20           # minimum raw samples to accept a breath

# Create output directories
for label in LABELS_USE:
    (OUT_ROOT / label).mkdir(parents=True, exist_ok=True)

print(f'Output root : {OUT_ROOT.resolve()}')
print(f'Image size  : {IMG_SIZE}×{IMG_SIZE} px')
print(f'Resample to : {N_SAMPLES} points per breath')

Output root : /Users/ashren/Desktop/uni/vandy/xai/project/breath_images
Image size  : 224×224 px
Resample to : 300 points per breath


## 2. Load all patient data

In [2]:
def load_patient(patient_dir: Path):
    """Load breath labels and signal parquet for one patient. Returns (breaths_df, signals_df, pid) or None."""
    pid = patient_dir.name

    csv_files  = list(patient_dir.glob('*breaths*.csv'))
    xlsx_files = list(patient_dir.glob('*breaths*.xlsx'))
    if csv_files:
        breaths = pd.read_csv(csv_files[0])
    elif xlsx_files:
        breaths = pd.read_excel(xlsx_files[0])
    else:
        return None

    breaths.columns = [c.lower().strip() for c in breaths.columns]
    breaths = breaths.rename(columns={'start (s)': 'start_s', 'end (s)': 'end_s', 'label': 'label'})
    for col in ('start_s', 'end_s', 'label'):
        if col not in breaths.columns:
            return None

    parquet_files = list(patient_dir.glob('*signals*.parquet'))
    if not parquet_files:
        return None

    signals = pd.read_parquet(parquet_files[0])
    if 'time_s' not in signals.columns or 'sig_flow' not in signals.columns:
        return None

    return breaths, signals, pid


patient_dirs = sorted([d for d in DATA_PATH.iterdir() if d.is_dir()])
print(f'Found {len(patient_dirs)} patient directories')

all_records = []
skipped = []
for p_dir in patient_dirs:
    result = load_patient(p_dir)
    if result is None:
        skipped.append(p_dir.name)
    else:
        all_records.append(result)

print(f'Loaded  : {len(all_records)} patients')
print(f'Skipped : {skipped}')

Found 87 patient directories
Loaded  : 84 patients
Skipped : ['KG_2023-09', 'MB_2023-10', 'RB_2021-08']


## 3. Compute global Y-axis range

To ensure every image encodes the same scale, we fix the Y-axis globally.  
We normalise each breath by its own PIF, then compute the 1st–99th percentile of all normalised samples across all FL/NFL breaths.  
This gives a robust fixed range that avoids outlier-driven clipping.

In [3]:
all_normalised_samples = []

for breaths_df, signals_df, pid in all_records:
    for _, row in breaths_df.iterrows():
        label = str(row['label']).strip()
        if label not in LABELS_USE:
            continue

        window = signals_df[
            (signals_df['time_s'] >= float(row['start_s'])) &
            (signals_df['time_s'] <  float(row['end_s']))
        ]
        if len(window) < MIN_SAMPLES:
            continue

        flow = window['sig_flow'].values.astype(np.float64)
        pif  = np.max(flow)
        if pif <= 0:
            continue

        all_normalised_samples.append(flow / pif)

all_vals = np.concatenate(all_normalised_samples)
Y_MIN = float(np.percentile(all_vals, 1))
Y_MAX = float(np.percentile(all_vals, 99))

# Add a small margin so the waveform doesn't touch the image edges
margin = 0.05 * (Y_MAX - Y_MIN)
Y_MIN -= margin
Y_MAX += margin

print(f'Global Y range (normalised by PIF): [{Y_MIN:.3f}, {Y_MAX:.3f}]')
print(f'Total breath windows sampled: {len(all_normalised_samples)}')

Global Y range (normalised by PIF): [-1.758, 1.125]
Total breath windows sampled: 7007


## 4. Generate and save breath images

In [4]:
def resample_breath(flow: np.ndarray, n: int) -> np.ndarray:
    """Resample a 1D flow signal to exactly `n` evenly-spaced points."""
    x_orig = np.linspace(0, 1, len(flow))
    x_new  = np.linspace(0, 1, n)
    return interp1d(x_orig, flow, kind='linear')(x_new)


def save_breath_image(flow_norm: np.ndarray, out_path: Path,
                      img_size: int, y_min: float, y_max: float) -> None:
    """
    Plot a single normalised breath waveform and save as a clean PNG.
    No axes, ticks, labels, or padding — just the waveform on a white background.
    """
    dpi = 100
    fig_size_in = img_size / dpi  # e.g. 2.24 inches for 224px at 100 dpi

    fig, ax = plt.subplots(figsize=(fig_size_in, fig_size_in), dpi=dpi)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    x = np.linspace(0, 1, len(flow_norm))
    ax.plot(x, flow_norm, color=LINE_COLOR, linewidth=LINE_WIDTH, solid_capstyle='round')

    ax.set_xlim(0, 1)
    ax.set_ylim(y_min, y_max)
    ax.axis('off')

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(out_path, dpi=dpi, bbox_inches='tight', pad_inches=0,
                facecolor='white')
    plt.close(fig)


# ── Main loop ─────────────────────────────────────────────────────────────────
counts   = {'FL': 0, 'NFL': 0}
skipped_breaths = 0

for breaths_df, signals_df, pid in all_records:
    # Use breath_number column if present; fall back to row index
    bn_col = None
    for candidate in ('breath #', 'breath_number', 'breath_num', 'breath #'):
        if candidate in breaths_df.columns:
            bn_col = candidate
            break

    for idx, row in breaths_df.iterrows():
        label = str(row['label']).strip()
        if label not in LABELS_USE:
            continue

        start_s = float(row['start_s'])
        end_s   = float(row['end_s'])

        window = signals_df[
            (signals_df['time_s'] >= start_s) &
            (signals_df['time_s'] <  end_s)
        ]
        if len(window) < MIN_SAMPLES:
            skipped_breaths += 1
            continue

        flow = window['sig_flow'].values.astype(np.float64)
        pif  = np.max(flow)
        if pif <= 0:
            skipped_breaths += 1
            continue

        # Normalise amplitude and resample to fixed length
        flow_norm = resample_breath(flow / pif, N_SAMPLES)

        # Build filename
        if bn_col is not None:
            bn = row[bn_col]
            if pd.notna(bn):
                bn = int(bn)
            else:
                bn = idx
        else:
            bn = idx

        # Sanitise participant ID for use in filename
        safe_pid = pid.replace(' ', '_').replace('(', '').replace(')', '')
        filename = f'{safe_pid}_breath_{bn}.png'
        out_path = OUT_ROOT / label / filename

        save_breath_image(flow_norm, out_path, IMG_SIZE, Y_MIN, Y_MAX)
        counts[label] += 1

print('=== Done ===')
for lbl, n in counts.items():
    print(f'  {lbl}: {n} images  →  {OUT_ROOT / lbl}')
print(f'  Skipped (too short / no positive flow): {skipped_breaths}')

=== Done ===
  FL: 3638 images  →  breath_images/FL
  NFL: 3369 images  →  breath_images/NFL
  Skipped (too short / no positive flow): 0


## 5. Quick sanity-check: preview a sample grid

In [5]:
import random
from PIL import Image

matplotlib.use('inline' if 'inline' in matplotlib.get_backend() else 'Agg')
import importlib
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

N_PREVIEW = 6   # images per class

fig, axes = plt.subplots(2, N_PREVIEW, figsize=(N_PREVIEW * 2.5, 5))

for row_idx, label in enumerate(['FL', 'NFL']):
    img_paths = sorted((OUT_ROOT / label).glob('*.png'))
    sample    = random.sample(img_paths, min(N_PREVIEW, len(img_paths)))
    for col_idx, p in enumerate(sample[:N_PREVIEW]):
        ax = axes[row_idx][col_idx]
        ax.imshow(Image.open(p), cmap='gray', vmin=0, vmax=255)
        ax.set_title(p.stem[:20], fontsize=6)
        ax.axis('off')
    axes[row_idx][0].set_ylabel(label, fontsize=11, fontweight='bold',
                                rotation=0, labelpad=40, va='center')

plt.suptitle('Sample breath images (FL top, NFL bottom)', fontweight='bold')
plt.tight_layout()
plt.savefig('breath_image_preview.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: breath_image_preview.png')

<Figure size 1500x500 with 12 Axes>

Saved: breath_image_preview.png


## 6. Dataset summary

In [6]:
fl_count  = len(list((OUT_ROOT / 'FL').glob('*.png')))
nfl_count = len(list((OUT_ROOT / 'NFL').glob('*.png')))
total     = fl_count + nfl_count

print('=== Breath Image Dataset Summary ===')
print(f'  FL images  : {fl_count}')
print(f'  NFL images : {nfl_count}')
print(f'  Total      : {total}')
print(f'  FL %       : {100 * fl_count / total:.1f}%')
print(f'  Image size : {IMG_SIZE}×{IMG_SIZE} px (white bg, black waveform)')
print(f'  Y range    : [{Y_MIN:.3f}, {Y_MAX:.3f}] (normalised by PIF, fixed across all images)')
print(f'  X range    : 0–1 (fractional breath time, {N_SAMPLES} resampled points)')
print(f'  Location   : {OUT_ROOT.resolve()}')

=== Breath Image Dataset Summary ===
  FL images  : 3638
  NFL images : 3369
  Total      : 7007
  FL %       : 51.9%
  Image size : 224×224 px (white bg, black waveform)
  Y range    : [-1.758, 1.125] (normalised by PIF, fixed across all images)
  X range    : 0–1 (fractional breath time, 300 resampled points)
  Location   : /Users/ashren/Desktop/uni/vandy/xai/project/breath_images
